### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [10]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from pathlib import Path

In [11]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Monteiro_2024_Environ._Res._Lett._19_124012.pdf
  ✓ Loaded 13 pages

Processing: s12992-020-00589-w.pdf
  ✓ Loaded 11 pages

Total documents loaded: 24


In [12]:
all_pdf_documents

[Document(metadata={'producer': 'XeLaTeX; modified using iText® 5.5.13.5 ©2000-2026 iText Group NV (IOP Publishing Ltd; licensed version)', 'creator': 'IOPP', 'creationdate': '2024-10-28T20:56:00+00:00', 'subject': 'Environmental Research Letters, 19 (2024) 124012. doi: 10.1088/1748-9326/ad8807', 'moddate': '2026-10-13T02:34:14+01:00', 'author': 'Estela A Monteiro,Yona Silvy,David Hohn,Friedrich A Burger,Thomas L Frölicher,Nadine Mengis', 'title': 'FROT: A Framework to comprehensively describe radiative contributions to temperature responses', 'keywords': 'radiative contribution,radiative forcing,temperature stabilisation,Earth system feedbacks,ambitious climate mitigation', 'source': '..\\data\\pdf\\Monteiro_2024_Environ._Res._Lett._19_124012.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'Monteiro_2024_Environ._Res._Lett._19_124012.pdf', 'file_type': 'pdf'}, page_content='LETTER • OPEN ACCESS\nFROT: A Framework to comprehensively describe\nradiative contributio

In [13]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [14]:
chunks=split_documents(all_pdf_documents)
chunks

Split 24 documents into 127 chunks

Example chunk:
Content: LETTER • OPEN ACCESS
FROT: A Framework to comprehensively describe
radiative contributions to temperature responses
To cite this article: Estela A Monteiro et al 2024 Environ. Res. Lett. 19 124012
 
V...
Metadata: {'producer': 'XeLaTeX; modified using iText® 5.5.13.5 ©2000-2026 iText Group NV (IOP Publishing Ltd; licensed version)', 'creator': 'IOPP', 'creationdate': '2024-10-28T20:56:00+00:00', 'subject': 'Environmental Research Letters, 19 (2024) 124012. doi: 10.1088/1748-9326/ad8807', 'moddate': '2026-10-13T02:34:14+01:00', 'author': 'Estela A Monteiro,Yona Silvy,David Hohn,Friedrich A Burger,Thomas L Frölicher,Nadine Mengis', 'title': 'FROT: A Framework to comprehensively describe radiative contributions to temperature responses', 'keywords': 'radiative contribution,radiative forcing,temperature stabilisation,Earth system feedbacks,ambitious climate mitigation', 'source': '..\\data\\pdf\\Monteiro_2024_Environ._Res._Lett._1

[Document(metadata={'producer': 'XeLaTeX; modified using iText® 5.5.13.5 ©2000-2026 iText Group NV (IOP Publishing Ltd; licensed version)', 'creator': 'IOPP', 'creationdate': '2024-10-28T20:56:00+00:00', 'subject': 'Environmental Research Letters, 19 (2024) 124012. doi: 10.1088/1748-9326/ad8807', 'moddate': '2026-10-13T02:34:14+01:00', 'author': 'Estela A Monteiro,Yona Silvy,David Hohn,Friedrich A Burger,Thomas L Frölicher,Nadine Mengis', 'title': 'FROT: A Framework to comprehensively describe radiative contributions to temperature responses', 'keywords': 'radiative contribution,radiative forcing,temperature stabilisation,Earth system feedbacks,ambitious climate mitigation', 'source': '..\\data\\pdf\\Monteiro_2024_Environ._Res._Lett._19_124012.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'Monteiro_2024_Environ._Res._Lett._19_124012.pdf', 'file_type': 'pdf'}, page_content='LETTER • OPEN ACCESS\nFROT: A Framework to comprehensively describe\nradiative contributio

### embedding And vectorStoreDB

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

C:\Users\tr538\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager

### VectorStore